<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>


# Lecture — Data Encoding

## Overview

Spark ML estimators read features from a single numeric vector, so categorical columns such as `Contract` or `PaymentMethod` must be encoded first. This lecture shows how to do that **on Databricks with Spark ML**: find the categorical columns, encode them in a pipeline that learns the categories from training data, and handle missing, unseen, and rare values the same way in training and in production.

This lecture covers the following sections:

- **A. Categorical Features on Databricks:** find categorical columns and their cardinality with Spark, and the three common challenges: high cardinality, missing values, and string types
- **B. One-Hot Encoding with Spark ML:** the standard `StringIndexer` → `OneHotEncoder` → `VectorAssembler` pipeline, what it produces, and its trade-offs
- **C. Label, Ordinal, and Target Encoding:** when integer indices are acceptable, explicit ordinal mappings, and Spark ML's `TargetEncoder` as an advanced option
- **D. Optional: Embeddings:** dense vector representations for very high-cardinality or unstructured features
- **E. Conclusion:** which encoding to start with, and key takeaways

## Learning Objectives

By the end of this lecture, you will be able to:

- Describe how to find categorical columns and their number of distinct values in a Spark DataFrame
- Distinguish nominal, ordinal, binary, and high-cardinality categorical features
- Explain how a Spark ML pipeline of `StringIndexer` → `OneHotEncoder` → `VectorAssembler` encodes categories learned from training data
- Explain how `handleInvalid="keep"` handles missing and unseen categories consistently during training and inference
- Explain how to group rare categories into an `"Other"` value using frequencies learned from training data
- Explain when to use an explicit ordinal mapping instead of `StringIndexer`
- Describe Spark ML's `TargetEncoder` and the leakage risk of target encoding
- Describe when embeddings may be useful as an advanced alternative

## A. Categorical Features on Databricks

### A1. Find the Categorical Features

Many Spark ML estimators expect all features in one numeric vector, so categorical values must be transformed first. Some libraries can handle categories natively, so the right encoding depends on the estimator, the feature, and how the model will be used in production.

Start by finding the categorical columns and how many distinct values each one has. On Databricks, the **Data Profile** (from `display(df)`) shows this visually, and Spark gives you exact numbers in code:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.sql import functions as F</code>
</br>
<code># String columns are the candidates for encoding (exclude IDs and the target)</code>
<code>category_cols = [f.name for f in telco_df.schema.fields</code>
<code>                 if f.dataType.simpleString() == &quot;string&quot; and f.name not in (&quot;customerID&quot;, &quot;Churn&quot;)]</code>
</br>
<code># How many distinct values does each one have?</code>
<code>display(telco_df.select([F.approx_count_distinct(c).alias(c) for c in category_cols]))</code></pre>
</div>

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Type</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Example</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Starting point on Databricks</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Nominal</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">City, PaymentMethod (no order)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>StringIndexer</code> → <code>OneHotEncoder</code></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Ordinal</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Small &lt; Medium &lt; Large</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">An explicit mapping you define (section C)</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Binary</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Yes / No</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A 0/1 column, or the same one-hot pipeline</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">High-cardinality</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Product ID, ZIP code</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Group rare values into <code>&quot;Other&quot;</code>; consider advanced encodings</td></tr></tbody></table></div>

### A2. Three Categorical-Feature Challenges

Categorical features present three recurring challenges in ML workflows. The table below summarizes each challenge, why it's difficult, and where it is typically addressed.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">The challenge</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">The issue</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">The solution</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">High cardinality</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Many unique values in a categorical feature can lead to increased dimensionality and potential overfitting.</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Group rare categories into "Other", keep the top N plus "Other", or use an advanced encoding (target encoding, embeddings)</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Missing values</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Categorical features often have missing values, which need to be addressed before model training</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Treat missing as its own category ("Unknown", or <code>StringIndexer(handleInvalid="keep")</code>), or impute the mode or a constant</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">String types</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Spark ML estimators read a numeric feature vector, so string categories must be encoded first.</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">One-hot encoding, label encoding, target encoding, embeddings</td></tr></tbody></table></div>

### A3. High Cardinality Issues and Solutions

**Issue: Large set of categories**

Many unique values in a categorical feature lead to large numbers of dummy variables, increased sparsity, and potential overfitting — especially when some categories appear only a few times.

**Possible Solutions:**

<div style="display: flex; gap: 32px; flex-wrap: wrap; margin: 16px 0;">

<!-- Left table: Group Rare Categories -->
<div style="flex: 1; min-width: 300px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 320px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Group Rare Categories</div>
<table style="border-collapse: separate; border-spacing: 0; width: 320px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Category</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Count</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Grouped</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">1</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1200</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">2</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">980</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">2</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">3</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">870</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">4</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">12</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">Other</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">8</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">Other</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">6</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">Other</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">Rare categories (count &lt; threshold) are grouped into one explicit value, <code>"Other"</code>. Choose the threshold from training-data frequencies.</p>
</div>

<!-- Right table: Top-N Categories -->
<div style="flex: 1; min-width: 300px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 320px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Top-N Categories</div>
<table style="border-collapse: separate; border-spacing: 0; width: 320px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Category</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Count</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Kept?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">1</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1200</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 600; color: #00A972; background: #E6F5EE;">✓ Top 3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">2</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">980</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 600; color: #00A972; background: #E6F5EE;">✓ Top 3</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">3</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">870</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 600; color: #00A972; background: #E6F5EE;">✓ Top 3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">4</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">12</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">→ Other</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">8</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">→ Other</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">6</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">→ Other</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">Only the top-N most frequent categories are kept as they are; the rest are mapped to <code>"Other"</code> rather than removed.</p>
</div>

</div>

On Databricks, learn the frequent categories from the **training data**, then apply the same mapping to every dataset:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>min_count = 100</code>
<code>frequent = [r[&quot;PaymentMethod&quot;] for r in</code>
<code>            train_df.groupBy(&quot;PaymentMethod&quot;).count().filter(F.col(&quot;count&quot;) &gt;= min_count).collect()</code>
<code>            if r[&quot;PaymentMethod&quot;] is not None]</code>
</br>
<code>def group_rare(df):</code>
<code>    return df.withColumn(</code>
<code>        &quot;PaymentMethod_grouped&quot;,</code>
<code>        F.when(F.col(&quot;PaymentMethod&quot;).isNull(), &quot;Unknown&quot;)</code>
<code>         .when(F.col(&quot;PaymentMethod&quot;).isin(frequent), F.col(&quot;PaymentMethod&quot;))</code>
<code>         .otherwise(&quot;Other&quot;),</code>
<code>    )</code></pre>
</div>

### A4. Missing Values Issues and Solutions

**Issue: Categorical gaps**

Categorical features often contain missing values, which need a strategy before encoding and model training. Unlike numeric columns, you cannot simply compute a mean — you need a strategy that respects the discrete nature of the data.

**Possible Solutions:**

<div style="display: flex; gap: 32px; flex-wrap: wrap; margin: 16px 0;">

<!-- Left table: Imputation -->
<div style="flex: 1; min-width: 300px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 320px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Imputation</div>
<table style="border-collapse: separate; border-spacing: 0; width: 320px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">ID</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Value</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">→</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Imputed</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">3</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">2</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">7</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">7</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">NaN</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">4</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">3</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">NaN</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">3</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">Replace missing values with the mode (most frequent value) or a constant.</p>
</div>

<!-- Right table: Consider Missing as a Separate Category -->
<div style="flex: 1; min-width: 300px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 320px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Missing as a Separate Category</div>
<table style="border-collapse: separate; border-spacing: 0; width: 320px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">ID</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Value</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">→</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Filled</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">3</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">2</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">7</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">7</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">NaN</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">Unknown</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">4</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">3</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; color: #618794;">NaN</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">Unknown</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">Treat missingness as informative — give it its own category, such as <code>"Unknown"</code>, then encode it like any other value. With <code>StringIndexer(handleInvalid="keep")</code>, nulls get their own index automatically.</p>
</div>

</div>

### A5. Encoding Issues and Solutions

**Issue: String types**

Models require numerical input, and categorical variables stored as strings must be converted to numbers before training. The choice of encoding method depends on the nature of the categories and the model being used.

**Possible Solutions:**

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 16px 0;">

<!-- Left table: One-Hot Encoding -->
<div style="flex: 1; min-width: 240px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 300px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">One-Hot Encoding</div>
<table style="border-collapse: separate; border-spacing: 0; width: 300px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">City</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">New York</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Paris</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">London</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">London</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">One binary column per category. No ordinal relationship implied.</p>
</div>

<!-- Middle table: Label Encoding -->
<div style="flex: 1; min-width: 200px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 240px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Label Encoding</div>
<table style="border-collapse: separate; border-spacing: 0; width: 240px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">City</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">→</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Label</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">0</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">London</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">2</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">0</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">Each category gets a unique integer. Simple but may imply false ordering.</p>
</div>

<!-- Right table: Ordinal Encoding -->
<div style="flex: 1; min-width: 200px; display: flex; flex-direction: column; align-items: center; text-align: center;">
<div style="width: 240px; text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Ordinal Encoding</div>
<table style="border-collapse: separate; border-spacing: 0; width: 240px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Size</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">→</th>
      <th style="padding: 10px 12px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Ordinal</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Small</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Medium</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">2</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Large</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">3</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Small</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Large</strong></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 10px 12px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">3</td>
    </tr>
  </tbody>
</table>
<p style="font-size: 12px; color: #618794; margin-top: 8px;">Like label encoding, but the order is meaningful (Small &lt; Medium &lt; Large).</p>
</div>

</div>

## B. One-Hot Encoding with Spark ML

### B1. The Spark ML Pattern: `StringIndexer` → `OneHotEncoder` → `VectorAssembler`

On Databricks, one-hot encoding is a Spark ML **pipeline** with three stages:

1. **`StringIndexer`** converts each category to a numeric index. It is an estimator: it learns the list of categories from the training data. By default the most frequent category gets index `0.0`.
2. **`OneHotEncoder`** converts each index to a sparse binary vector. It also learns from the training data how many categories each column has.
3. **`VectorAssembler`** combines the encoded columns into one vector for the model.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.ml import Pipeline</code>
<code>from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler</code>
</br>
<code>category_cols = [&quot;gender&quot;, &quot;Contract&quot;, &quot;PaymentMethod&quot;]</code>
<code>index_cols = [f&quot;{c}_idx&quot; for c in category_cols]</code>
<code>ohe_cols = [f&quot;{c}_ohe&quot; for c in category_cols]</code>
</br>
<code>pipeline = Pipeline(stages=[</code>
<code>    StringIndexer(inputCols=category_cols, outputCols=index_cols, handleInvalid=&quot;keep&quot;),</code>
<code>    OneHotEncoder(inputCols=index_cols, outputCols=ohe_cols),       # dropLast=True by default</code>
<code>    VectorAssembler(inputCols=ohe_cols, outputCol=&quot;categorical_features&quot;),</code>
<code>])</code>
</br>
<code>pipeline_model = pipeline.fit(train_df)          # category lists are learned from training data only</code>
<code>test_encoded = pipeline_model.transform(test_df)</code>
</br>
<code># A value that never appeared in training still gets an index instead of an error</code>
<code>new_df = spark.createDataFrame([(&quot;Female&quot;, &quot;Month-to-month&quot;, &quot;Crypto wallet&quot;)], category_cols)</code>
<code>display(pipeline_model.transform(new_df).select(&quot;PaymentMethod&quot;, &quot;PaymentMethod_idx&quot;, &quot;PaymentMethod_ohe&quot;))</code></pre>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #0d47a1;">Info</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><code>StringIndexer(handleInvalid=&quot;keep&quot;)</code> assigns null values and categories that were not seen during training to one extra index. Without it, an unseen value at inference time raises an error. Because <code>OneHotEncoder</code> drops the last position by default, and that last position is this extra index, a null or unseen value such as <code>"Crypto wallet"</code> is encoded as an <b>all-zero vector</b>. That is the empty vector you see in the example output. Because the fitted pipeline is reused for test and production data, missing and unseen values are handled the same way everywhere.</div></div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note: dropLast</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><code>OneHotEncoder</code> leaves out the last category by default (<code>dropLast=True</code>), because it is implied when all other positions are 0. This avoids a redundant column, which matters for linear models with an intercept, and it does not lose information when categories are mutually exclusive. Set <code>dropLast=False</code> only when every category must have its own explicit position.</div></div>

<div style="border-left: 4px solid #00A972; background: #E6F5EE; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #00805A;">Databricks tip</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Save the fitted pipeline to a Unity Catalog volume and reuse it for every dataset and for inference. You build a full pipeline like this in the <i>2.6 Demo</i>.</div></div>

### B2. What One-Hot Encoding Produces

One-hot encoding represents each category as its own position. Each row has a `1` in exactly one position and `0` in the rest. The table shows every category for clarity; in Spark, the last one is left out by default (`dropLast=True`), and the result is stored as a sparse vector.

<table style="border-collapse: separate; border-spacing: 0; width: 100%; max-width: 680px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">ID</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">City</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">→</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">New York</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Paris</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">London</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">2</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">London</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">4</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">5</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">0</td>
    </tr>
  </tbody>
</table>

### B3. Trade-offs

While one-hot encoding is widely used, it introduces several practical limitations:

- **Sparsity** — most entries are zero; Spark stores the result as sparse vectors, which keeps memory use low, but the number of model inputs still grows
- **Poor scaling with cardinality** — feature space explodes as the number of categories grows
- **Obscured feature importance** — a single concept is split across many columns

## C. Label, Ordinal, and Target Encoding

### C1. Label Encoding (String Indexing)

`StringIndexer` assigns each category an integer **by frequency** — the most common category gets index 0, the next most common gets 1, and so on. The indices have no ordinal meaning.

<table style="border-collapse: separate; border-spacing: 0; width: 100%; max-width: 600px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">ID</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Contract</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">→</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Contract_idx</th>
      <th style="padding: 12px 16px; text-align: center; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Why this index?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Month-to-month</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-size: 12px; color: #618794;">Most frequent</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">2</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">One year</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">2</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-size: 12px; color: #618794;">Least frequent</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">3</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Two year</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-size: 12px; color: #618794;">2nd most frequent</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;">4</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Month-to-month</strong></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9;"></td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">0</td>
      <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #EEEDE9; font-size: 12px; color: #618794;">Same as row 1</td>
    </tr>
  </tbody>
</table>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 10px 16px; cursor: pointer; font-weight: 600; font-size: 13pt; border-radius: 8px; user-select: none;">
When to use / not use StringIndexer for label encoding
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 12px 16px; background: #F9F7F4; font-size: 13pt; line-height: 1.6; color: #1B3139;">
<ul style="margin: 4px 0 0 16px;">
<li><strong>When it can work</strong> — Spark ML tree models (decision trees, Random Forest, GBT) read the category metadata that <code>StringIndexer</code> attaches to its output, so they can treat the indices as <b>categories</b> and split on groups of categories rather than on numeric thresholds. Set <code>maxBins</code> to at least the number of categories in the largest column. Check the result against one-hot encoding with validation results.</li>
<li><strong>Important nuance</strong> — This relies on the column metadata. If the metadata is lost (for example, if you rebuild the column with a plain cast) or you use a library that ignores it, the indices are read as ordered numbers. This guidance is specific to Spark ML tree models and should not be generalized to sklearn-style pipelines.</li>
<li><strong>When NOT to use</strong> — Linear models will infer a numeric relationship between indices (0 &lt; 1 &lt; 2) that doesn't exist. Outside Spark ML, other tree libraries may prefer one-hot encoding or native categorical support instead.</li>
</ul>
</div>
</details>
<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Warning</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><b>Ordinal encoding is <u>not</u> handled by StringIndexer.</b><br>
    If your categories have a true order (e.g., Freshman &lt; Sophomore &lt; Junior &lt; Senior), <b>do not use StringIndexer</b>—it assigns indices by frequency, not order. You must map ordered categories manually (e.g., with <code>when().otherwise()</code> or a mapping DataFrame). Using StringIndexer here can mislead your model and invalidate results.</div></div>

For a feature with a **real order**, define the order yourself instead of relying on `StringIndexer`'s frequency order:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>contract_ord = (</code>
<code>    F.when(F.col(&quot;Contract&quot;) == &quot;Month-to-month&quot;, 1)</code>
<code>     .when(F.col(&quot;Contract&quot;) == &quot;One year&quot;, 2)</code>
<code>     .when(F.col(&quot;Contract&quot;) == &quot;Two year&quot;, 3)</code>
<code>)</code>
<code>telco_df = telco_df.withColumn(&quot;Contract_ord&quot;, contract_ord)</code></pre>
</div>

The values 1, 2, 3 imply equal steps between categories, which may not be true. Use an ordinal mapping only when the order is meaningful for the model.

### C2. Target Encoding (Advanced)

Target encoding replaces each category with the **mean of the target variable** for all rows belonging to that category.

<table style="border-collapse: separate; border-spacing: 0; width: 100%; max-width: 540px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">City</th>
      <th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Target (Clicked)</th>
      <th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Mean</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">New York</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">[1, 0]</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">0.5</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Paris</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">[0, 1]</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">0.5</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">London</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">[1]</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; font-weight: 700; color: #FF5F46; background: #FFF1EE;">1.0</td>
    </tr>
  </tbody>
</table>

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Warning</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><b>Target leakage is a critical risk with target encoding.</b><br>
    Never compute category means using the full dataset (including validation or test data)—this leaks future information into your training features and invalidates model evaluation. <br>
    <b>Always compute means on the training set only</b>, then apply those means to validation and test sets. For categories not seen during training, use the global training-set target mean as a fallback.</div></div>

**Other drawbacks:**
- **Overfitting risk** — small categories yield over-confident mean estimates
- **Target distribution dependency** — concept drift in the target invalidates stored means

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Spark ML includes a built-in <code>TargetEncoder</code> (<code>pyspark.ml.feature.TargetEncoder</code>, available since Apache Spark 4.0, so it works on Serverless V5). It expects category <b>indices</b> as input, so run <code>StringIndexer</code> first. Set <code>targetType="binary"</code> for classification or <code>"continuous"</code> for regression, use <code>smoothing</code> to blend small categories toward the overall mean, and set <code>handleInvalid="keep"</code> to map unseen categories to the overall statistic. As with any encoder that learns from the label, <b>fit it on the training set only</b> to avoid target leakage.<pre style="background:#fff; border:1px solid #DCE0E2; border-radius:6px; padding:10px 14px; font-size:11pt; margin-top:10px;">from pyspark.ml.feature import StringIndexer, TargetEncoder
&nbsp;
indexer = StringIndexer(inputCol="PaymentMethod", outputCol="PaymentMethod_idx")
encoder = TargetEncoder(inputCol="PaymentMethod_idx", outputCol="PaymentMethod_te",
                        labelCol="label", targetType="binary", smoothing=1.0, handleInvalid="keep")
# Fit both stages on train_df only, then transform train_df and test_df</pre></div></div>

## D. Optional: Embeddings

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Caution</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">This section is optional background. Embeddings are most useful for very high-cardinality features or for text, images, and other unstructured data. For typical tabular categories, start with the Spark ML pipeline in section B.</div></div>

### D1. What Embeddings Are

**Embeddings** are dense, low-dimensional vector representations of data. Unlike one-hot encoding's sparse, high-dimensional binary vectors, embeddings can place **items that behave similarly close together** in a continuous vector space. How close two items end up depends on the data and the task the embedding was trained for.

Embeddings are used widely in **text, categorical features, images, and audio**. The table below contrasts one-hot and embedding representations for a small product catalog:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Item</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">One-Hot Vector</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Embedding Vector</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Note</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">TV</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[1, 0, 0, 0]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0.5, 1.2, -0.8]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Laptop</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0, 1, 0, 0]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0.6, 1.1, -0.7]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Close to TV in embedding space</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Book</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0, 0, 1, 0]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0.2, -0.4, 1.5]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Mug</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0, 0, 0, 1]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>[0.7, -0.5, 0.4]</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #5A6F77; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F0F2F4; color: #5A6F77; font-weight: 700; border: 1px solid #5A6F7740; white-space: nowrap;">Dimensions</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">4 — one per category</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">3 — fixed regardless of catalog size</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Embeddings scale better with cardinality</td></tr></tbody></table></div>

The scatter plot below illustrates how an embedding can place **similar items close together** in a 2D projection of the vector space. TV and Laptop (both electronics with screens) cluster nearby, while Book and Mug occupy distinct regions.

<div style="max-width: 500px; margin: 24px auto;">
<svg width="100%" viewBox="0 0 460 360" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Embedding Space Visualization</title>
  <desc>TV and Laptop are close together in embedding space; Book and Mug are far apart</desc>
  <!-- Background -->
  <rect x="40" y="20" width="400" height="300" rx="8" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
  <!-- Grid lines -->
  <line x1="140" y1="20" x2="140" y2="320" stroke="#EEEDE9" stroke-width="0.5" stroke-dasharray="4 3"/>
  <line x1="240" y1="20" x2="240" y2="320" stroke="#EEEDE9" stroke-width="0.5" stroke-dasharray="4 3"/>
  <line x1="340" y1="20" x2="340" y2="320" stroke="#EEEDE9" stroke-width="0.5" stroke-dasharray="4 3"/>
  <line x1="40" y1="95" x2="440" y2="95" stroke="#EEEDE9" stroke-width="0.5" stroke-dasharray="4 3"/>
  <line x1="40" y1="170" x2="440" y2="170" stroke="#EEEDE9" stroke-width="0.5" stroke-dasharray="4 3"/>
  <line x1="40" y1="245" x2="440" y2="245" stroke="#EEEDE9" stroke-width="0.5" stroke-dasharray="4 3"/>
  <!-- Axes -->
  <line x1="40" y1="320" x2="440" y2="320" stroke="#1B3139" stroke-width="1.5"/>
  <line x1="40" y1="20" x2="40" y2="320" stroke="#1B3139" stroke-width="1.5"/>
  <text x="240" y="348" text-anchor="middle" font-size="12" fill="#618794">Dimension 1</text>
  <text x="16" y="170" text-anchor="middle" font-size="12" fill="#618794" transform="rotate(-90, 16, 170)">Dimension 2</text>
  <!-- Cluster highlight: TV + Laptop -->
  <ellipse cx="320" cy="80" rx="65" ry="45" fill="#2272B4" fill-opacity="0.08" stroke="#2272B4" stroke-width="1" stroke-dasharray="4 2"/>
  <!-- TV point -->
  <circle cx="300" cy="70" r="8" fill="#FF5F46" stroke="#1B3139" stroke-width="1.5"/>
  <text x="300" y="55" text-anchor="middle" font-size="13" font-weight="600" fill="#1B3139">TV</text>
  <!-- Laptop point (close to TV) -->
  <circle cx="340" cy="90" r="8" fill="#FF5F46" stroke="#1B3139" stroke-width="1.5"/>
  <text x="358" y="82" text-anchor="start" font-size="13" font-weight="600" fill="#1B3139">Laptop</text>
  <!-- Book point (far from electronics) -->
  <circle cx="120" cy="220" r="8" fill="#2272B4" stroke="#1B3139" stroke-width="1.5"/>
  <text x="120" y="245" text-anchor="middle" font-size="13" font-weight="600" fill="#1B3139">Book</text>
  <!-- Mug point (lower right: high dim1, low dim2) -->
  <circle cx="385" cy="235" r="8" fill="#00A972" stroke="#1B3139" stroke-width="1.5"/>
  <text x="385" y="258" text-anchor="middle" font-size="13" font-weight="600" fill="#1B3139">Mug</text>
  <!-- Distance annotation -->
  <line x1="305" y1="73" x2="335" y2="87" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="3 2"/>
  <text x="330" y="110" text-anchor="middle" font-size="11" fill="#618794" font-style="italic">close = similar</text>
</svg>
</div>

### D2. How Embeddings Are Produced

Embeddings can be sourced in three ways. Once produced, they are typically persisted as features, indexed for similarity search, and served in real time.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Source</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Examples</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Learned during training</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Neural network embedding layers — common for recommendation systems and deep tabular models</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Pretrained models</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Word2Vec, FastText, sentence transformers, image encoders — pre-computed on large corpora</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Foundation model APIs</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Embedding endpoints on Databricks Model Serving</td></tr></tbody></table></div>

Once you have embeddings, the typical workflow is:

1. **Persist them as features** in Feature Engineering in Unity Catalog
2. **Index them** in Databricks AI Search for similarity lookups
3. **Serve them in real time** via Feature Serving or Model Serving for downstream models


### D3. Categorical Embedding

**Transforming categorical variables into meaningful continuous vectors**

Categorical embeddings replace standard encoding methods with learned dense vectors. They can capture relationships between categories when the training task gives similar categories similar behavior.

**Steps:**
1. Convert categorical variables to integer indices
2. Pass through an embedding layer (a learned lookup table)
3. The result is a dense vector representation for each category

<div style="display: flex; align-items: center; gap: 8px; flex-wrap: wrap; max-width: 900px; margin: 24px auto;">

<!-- Step 1: Categorical Input -->
<div style="flex: 0 0 auto;">
<svg width="150" viewBox="0 0 150 260" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="75" y="16" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Categorical Input</text>
  <rect x="5" y="24" width="140" height="220" rx="8" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="20" y="44" width="110" height="32" rx="4" fill="#ffffff" stroke="#2272B4" stroke-width="1.2"/>
  <text x="75" y="65" text-anchor="middle" font-size="12" font-weight="600" fill="#2272B4">New York</text>
  <rect x="20" y="86" width="110" height="32" rx="4" fill="#ffffff" stroke="#2272B4" stroke-width="1.2"/>
  <text x="75" y="107" text-anchor="middle" font-size="12" font-weight="600" fill="#2272B4">Paris</text>
  <rect x="20" y="128" width="110" height="32" rx="4" fill="#ffffff" stroke="#2272B4" stroke-width="1.2"/>
  <text x="75" y="149" text-anchor="middle" font-size="12" font-weight="600" fill="#2272B4">London</text>
  <rect x="20" y="170" width="110" height="32" rx="4" fill="#ffffff" stroke="#2272B4" stroke-width="1.2"/>
  <text x="75" y="191" text-anchor="middle" font-size="12" font-weight="600" fill="#2272B4">Toronto</text>
</svg>
</div>

<!-- Arrow 1 -->
<div style="flex: 0 0 auto; font-size: 24px; color: #1B3139;">→</div>

<!-- Step 2: Integer Indices -->
<div style="flex: 0 0 auto;">
<svg width="100" viewBox="0 0 100 260" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="50" y="16" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Index</text>
  <rect x="5" y="24" width="90" height="220" rx="8" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="15" y="44" width="70" height="32" rx="4" fill="#ffffff" stroke="#FF5F46" stroke-width="1.2"/>
  <text x="50" y="65" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">0</text>
  <rect x="15" y="86" width="70" height="32" rx="4" fill="#ffffff" stroke="#FF5F46" stroke-width="1.2"/>
  <text x="50" y="107" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">1</text>
  <rect x="15" y="128" width="70" height="32" rx="4" fill="#ffffff" stroke="#FF5F46" stroke-width="1.2"/>
  <text x="50" y="149" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">2</text>
  <rect x="15" y="170" width="70" height="32" rx="4" fill="#ffffff" stroke="#FF5F46" stroke-width="1.2"/>
  <text x="50" y="191" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">3</text>
</svg>
</div>

<!-- Arrow 2 -->
<div style="flex: 0 0 auto; font-size: 24px; color: #1B3139;">→</div>

<!-- Step 3: Embedding Layer -->
<div style="flex: 0 0 auto;">
<svg width="140" viewBox="0 0 140 260" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="70" y="16" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Embedding Layer</text>
  <rect x="5" y="24" width="130" height="220" rx="8" fill="#2272B4" fill-opacity="0.08" stroke="#2272B4" stroke-width="1.5"/>
  <text x="70" y="50" text-anchor="middle" font-size="10" fill="#618794">(learned lookup table)</text>
  <rect x="15" y="62" width="110" height="26" rx="3" fill="#ffffff" stroke="#2272B4" stroke-width="1"/>
  <text x="70" y="80" text-anchor="middle" font-size="10" fill="#1B3139">idx 0 → [w₀₁, w₀₂, w₀₃]</text>
  <rect x="15" y="96" width="110" height="26" rx="3" fill="#ffffff" stroke="#2272B4" stroke-width="1"/>
  <text x="70" y="114" text-anchor="middle" font-size="10" fill="#1B3139">idx 1 → [w₁₁, w₁₂, w₁₃]</text>
  <rect x="15" y="130" width="110" height="26" rx="3" fill="#ffffff" stroke="#2272B4" stroke-width="1"/>
  <text x="70" y="148" text-anchor="middle" font-size="10" fill="#1B3139">idx 2 → [w₂₁, w₂₂, w₂₃]</text>
  <rect x="15" y="164" width="110" height="26" rx="3" fill="#ffffff" stroke="#2272B4" stroke-width="1"/>
  <text x="70" y="182" text-anchor="middle" font-size="10" fill="#1B3139">idx 3 → [w₃₁, w₃₂, w₃₃]</text>
  <text x="70" y="218" text-anchor="middle" font-size="10" font-style="italic" fill="#618794">Weights learned</text>
  <text x="70" y="234" text-anchor="middle" font-size="10" font-style="italic" fill="#618794">during training</text>
</svg>
</div>

<!-- Arrow 3 -->
<div style="flex: 0 0 auto; font-size: 24px; color: #1B3139;">→</div>

<!-- Step 4: Dense Vectors Output -->
<div style="flex: 0 0 auto;">
<svg width="180" viewBox="0 0 180 260" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="90" y="16" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Dense Vectors</text>
  <rect x="5" y="24" width="170" height="220" rx="8" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="15" y="44" width="150" height="32" rx="4" fill="#ffffff" stroke="#00A972" stroke-width="1.2"/>
  <text x="90" y="65" text-anchor="middle" font-size="11" font-weight="600" fill="#1B3139">[0.6, 1.2, -0.8]</text>
  <rect x="15" y="86" width="150" height="32" rx="4" fill="#ffffff" stroke="#00A972" stroke-width="1.2"/>
  <text x="90" y="107" text-anchor="middle" font-size="11" font-weight="600" fill="#1B3139">[0.3, -0.5, 0.9]</text>
  <rect x="15" y="128" width="150" height="32" rx="4" fill="#ffffff" stroke="#00A972" stroke-width="1.2"/>
  <text x="90" y="149" text-anchor="middle" font-size="11" font-weight="600" fill="#1B3139">[-0.2, 0.8, 0.4]</text>
  <rect x="15" y="170" width="150" height="32" rx="4" fill="#ffffff" stroke="#00A972" stroke-width="1.2"/>
  <text x="90" y="191" text-anchor="middle" font-size="11" font-weight="600" fill="#1B3139">[-0.9, 0.1, 1.3]</text>
</svg>
</div>

</div>

<p style="text-align: center; font-size: 12px; color: #618794; margin-top: 4px;">Each category gets a unique index and a learned dense vector (e.g. Paris → 1 → [0.3, -0.5, 0.9])</p>

### D4. Embeddings vs. One-Hot Encoding

The table below compares the two approaches across key properties to help you choose the right encoding for your use case.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Property</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">One-Hot Encoding</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Embeddings</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Representation</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Sparse binary vectors (high-dimensional)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Dense numeric vectors (low-dimensional)</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Captures semantics?</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">No</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Can — items that behave similarly in the training task tend to end up close together</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Scales with cardinality</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Poorly</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Well</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Best for</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Simple models, low-cardinality features</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Neural networks, large vocabularies, recommender systems</td></tr></tbody></table></div>

**Rule of thumb:**
- **One-hot encoding** — the default starting point for low- and moderate-cardinality nominal features with classical models; there is no fixed category limit, and Spark handles sparse vectors efficiently
- **Explicit ordinal mapping** — features with a meaningful natural order
- **Embeddings** — high-cardinality categoricals (IDs, products, locations) or any text and image features

## E. Conclusion

In this lecture, you learned how to encode categorical features on Databricks:

1. **Find categorical features** and their number of distinct values with the Data Profile and Spark, and classify them as nominal, ordinal, binary, or high-cardinality.
2. **Use the Spark ML pipeline** `StringIndexer` → `OneHotEncoder` → `VectorAssembler` as the default. It learns the categories from training data, and the fitted pipeline encodes test and inference data the same way.
3. **Handle missing and unseen categories** with `handleInvalid="keep"`, and group rare categories into `"Other"` using frequencies from the training data.
4. **Use an explicit ordinal mapping** only when the order is meaningful, and treat category indices fed directly to tree models as a trade-off to validate.
5. **Target encoding** (Spark ML `TargetEncoder`) is an advanced option that must be fit on training data only to avoid leakage.
6. **Embeddings** are an optional, advanced representation for very high-cardinality or unstructured features.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>